In [ ]:
# Установка необходимых библиотек
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

print("Библиотеки загружены.")

In [ ]:
# Пути к файлам
import os

path_amplitude_map = os.path.join("data", "amplitude_map.txt")
path_well_data = os.path.join("data", "well_data.txt")

try:
    df_map = pd.read_csv(path_amplitude_map)
    df_wells = pd.read_csv(path_well_data)
    print(f"✅ Все файлы успешно загружены. Строк в карте: {len(df_map)}, в скважинах: {len(df_wells)}")
except FileNotFoundError:
    print("❌ Ошибка: Файлы не найдены. Убедитесь, что они лежат в папке 'data'.")
except Exception as e:
    print(f"❌ Непредвиденная ошибка: {e}")

In [ ]:
# Загрузка данных
df_map = pd.read_csv(path_amplitude_map, sep=r'\s+')
df_wells = pd.read_csv(path_well_data, sep='\t')

# Переименовываем столбцы в amplitude_map.txt
df_map.columns = ['X', 'Y', 'amplitude']  # Amp -> amplitude
# Для df_wells уже правильные

# Разделение на train/test
from sklearn.model_selection import train_test_split
df_train, df_test_full = train_test_split(df_wells, test_size=0.2, random_state=42)
print(f"Размер обучающего набора: {len(df_train)} строк")
print(f"Размер тестового набора: {len(df_test_full)} строк")
print(f"Размер карты для предсказания: {len(df_map)} строк")

In [ ]:
# Подготовка данных
X_train = df_train[['amplitude']].copy()
y_train = df_train['thickness_m'].copy()

X_test_eval = df_test_full[['amplitude']].copy()
y_true = df_test_full['thickness_m'].copy()
X_test_pred = df_map[['amplitude']].copy()

print(f"Размер X_train: {X_train.shape}")
print(f"Пример первых строк X_train:")
print(X_train.head())

In [ ]:
# Нормализация данных
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_eval_scaled = scaler.transform(X_test_eval)
X_test_pred_scaled = scaler.transform(X_test_pred)

print("Данные нормализованы.")

In [ ]:
print("Обучение: Линейная регрессия...")

start_time = time.time()

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test_eval)
pred_lr_new = model_lr.predict(X_test_pred)

time_lr = time.time() - start_time
print(f"✅ Обучено за {time_lr:.4f} с")

In [ ]:
print("Обучение: Случайный лес...")

start_time = time.time()

model_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test_eval)
pred_rf_new = model_rf.predict(X_test_pred)

time_rf = time.time() - start_time
print(f"✅ Обучено за {time_rf:.4f} с")

In [ ]:
print("Обучение: Нейронная сеть...")

start_time = time.time()

# Масштабируем целевую переменную
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

model_mlp = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    alpha=0.01,
    learning_rate='adaptive',
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50
)

model_mlp.fit(X_train_scaled, y_train_scaled)

# Предсказания
pred_mlp = y_scaler.inverse_transform(
    model_mlp.predict(X_test_eval_scaled).reshape(-1, 1)
).ravel()

pred_mlp_new = y_scaler.inverse_transform(
    model_mlp.predict(X_test_pred_scaled).reshape(-1, 1)
).ravel()

time_mlp = time.time() - start_time
print(f"✅ Обучено за {time_mlp:.4f} с")

In [ ]:
def evaluate_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n{name}:")
    print(f" MAE: {mae:.4f}")
    print(f" RMSE: {rmse:.4f}")
    print(f" R²: {r2:.4f}")
    return mae, rmse, r2
    
print("Оценка моделей на тестовом наборе (20% well_data):")
results = {}
results['Линейная регрессия'] = evaluate_model(y_true, pred_lr, "Линейная регрессия")
results['Случайный лес'] = evaluate_model(y_true, pred_rf, "Случайный лес")
results['Нейронная сеть'] = evaluate_model(y_true, pred_mlp, "Нейронная сеть")

In [ ]:
print("\n⏱️ Время обучения:")
print(f"Линейная регрессия: {time_lr:.4f} с")
print(f"Случайный лес: {time_rf:.4f} с")
print(f"Нейронная сеть: {time_mlp:.4f} с")

fastest = min([('ЛинРег', time_lr), ('СлучЛес', time_rf), ('MLP', time_mlp)], key=lambda x: x[1])[0]
print(f"\n⚡ Самая быстрая модель: {fastest}")

In [ ]:
# Создаём DataFrame и сортируем по Well_Number
plot_data = pd.DataFrame({
    'Well_Number': range(1, len(df_test_full) + 1),
    'True_thickness': y_true,
    'LR': pred_lr,
    'RF': pred_rf,
    'MLP': pred_mlp
}).sort_values('Well_Number').reset_index(drop=True)

well_num = plot_data['Well_Number']
true_sorted = plot_data['True_thickness']
lr_sorted = plot_data['LR']
rf_sorted = plot_data['RF']
mlp_sorted = plot_data['MLP']

plt.figure(figsize=(14, 7))
plt.plot(well_num, true_sorted, label='Истинная мощность', color='black', linewidth=2.5)
plt.plot(well_num, lr_sorted, label='Линейная регрессия', linestyle='--')
plt.plot(well_num, rf_sorted, label='Случайный лес', linestyle='--')
plt.plot(well_num, mlp_sorted, label='Нейронная сеть', linestyle='--')

plt.xlabel('Номер скважины')
plt.ylabel('Мощность, м')
plt.title('Сравнение предсказаний с истинными значениями\n(обучение: 80% well_data, оценка: 20% well_data)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
pred_results = pd.DataFrame({
    'X': df_map['X'],
    'Y': df_map['Y'],
    'thickness_LR': pred_lr_new,
    'thickness_RF': pred_rf_new,
    'thickness_MLP': pred_mlp_new
})

output_path = "thickness_predictions.xlsx"
try:
    pred_results.to_excel(output_path, index=False)
    print(f"✅ Предсказания сохранены: {output_path}")
except Exception as e:
    print(f"❌ Ошибка сохранения: {e}")

In [ ]:
best_by_mae = min(results.items(), key=lambda x: x[1][0])[0]

print("\n" + "="*60)
print("ОТЧЁТ")
print("="*60)
print(f"🎯 Задача: обучиться на well_data → предсказать amplitude_map")
print(f"📊 Диапазон: вся площадь")
print(f"📌 Признаки: amplitude")
print(f"\n🏆 Наилучшая модель по MAE: {best_by_mae}")
print(f"⚡ Самая быстрая модель: {fastest}")
print(f"\n💾 Результаты: {output_path}")
print("="*60)